# HW4 Code - Decision Trees, Ensembles, and Naive Bayes

This notebook reproduces the analysis for HW4 using the local copies of:

- `spambase.data`
- `spambase.names`
- `agaricus-lepiota.data`

It generates:
- decision tree metrics
- pruning depth plot
- random forest metrics + feature importance plot
- AdaBoost metrics + ROC curve plot
- custom Naive Bayes results on Mushroom
- package Naive Bayes comparison


In [ ]:

import math
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

OUTPUT_DIR = "hw4_outputs"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Load datasets

In [ ]:

spambase = pd.read_csv("spambase.data", header=None)
X_spam = spambase.iloc[:, :-1]
y_spam = spambase.iloc[:, -1]

mushroom = pd.read_csv("agaricus-lepiota.data", header=None)
X_mush_raw = mushroom.iloc[:, 1:]
y_mush_raw = mushroom.iloc[:, 0]

print("SPAMBASE shape:", spambase.shape)
print("Mushroom shape:", mushroom.shape)


## Helper functions

In [ ]:

def compute_metrics(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    results = {}
    for split_name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
        y_pred = model.predict(X)
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X)[:, 1]
        else:
            y_score = model.decision_function(X)
        results[split_name] = {
            "error": 1 - accuracy_score(y, y_pred),
            "accuracy": accuracy_score(y, y_pred),
            "f1": f1_score(y, y_pred),
            "auc": roc_auc_score(y, y_score),
        }
    return results


## Problem 1 - Decision Trees

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_spam, y_spam, test_size=0.25, random_state=42, stratify=y_spam
)

dt_entropy = compute_metrics(
    DecisionTreeClassifier(criterion="entropy", random_state=42),
    X_train, y_train, X_test, y_test
)
dt_gini = compute_metrics(
    DecisionTreeClassifier(criterion="gini", random_state=42),
    X_train, y_train, X_test, y_test
)

dt_rows = []
for model_name, res in [("Entropy", dt_entropy), ("Gini", dt_gini)]:
    for split in ["train", "test"]:
        r = res[split]
        dt_rows.append([model_name, split, r["error"], r["accuracy"], r["f1"], r["auc"]])

dt_table = pd.DataFrame(dt_rows, columns=["Model", "Split", "Error", "Accuracy", "F1", "AUC"])
dt_table.round(4)


In [ ]:

depth_rows = []
for depth in range(1, 31):
    model = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_error = 1 - accuracy_score(y_train, model.predict(X_train))
    test_error = 1 - accuracy_score(y_test, model.predict(X_test))
    depth_rows.append([depth, train_error, test_error])

depth_df = pd.DataFrame(depth_rows, columns=["depth", "train_error", "test_error"])
best_depth = int(depth_df.loc[depth_df["test_error"].idxmin(), "depth"])
print("Recommended depth:", best_depth)

plt.figure(figsize=(7, 4.5))
plt.plot(depth_df["depth"], depth_df["train_error"], marker="o", label="Training error")
plt.plot(depth_df["depth"], depth_df["test_error"], marker="s", label="Testing error")
plt.xlabel("Maximum tree depth")
plt.ylabel("Classification error")
plt.title("Decision tree pruning: error vs. maximum depth")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "depth_vs_error.png"), dpi=200)
plt.show()


## Problem 2 - Random Forest

In [ ]:

rf_results = {}
for T in [10, 50, 100, 500]:
    rf_results[T] = compute_metrics(
        RandomForestClassifier(n_estimators=T, random_state=42),
        X_train, y_train, X_test, y_test
    )

rf_rows = []
for T, res in rf_results.items():
    for split in ["train", "test"]:
        r = res[split]
        rf_rows.append([T, split, r["accuracy"], r["f1"], r["auc"]])

rf_table = pd.DataFrame(rf_rows, columns=["Trees", "Split", "Accuracy", "F1", "AUC"])
rf_table.round(4)


In [ ]:

feature_names = []
with open("spambase.names", "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        if ":" in line and ("continuous" in line or "nominal" in line):
            name = line.split(":")[0].strip()
            if name != "spam":
                feature_names.append(name)

rf100 = RandomForestClassifier(n_estimators=100, random_state=42)
rf100.fit(X_train, y_train)
importances = pd.Series(rf100.feature_importances_, index=feature_names).sort_values(ascending=False)

top15 = importances.head(15)
plt.figure(figsize=(8, 5.5))
top15.sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest (100 trees): top 15 feature importances")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "rf_feature_importance.png"), dpi=200)
plt.show()

top15


## Problem 3 - AdaBoost

In [ ]:

ada_results = {}
for T in [10, 50, 100, 500]:
    ada_results[T] = compute_metrics(
        AdaBoostClassifier(n_estimators=T, random_state=42),
        X_train, y_train, X_test, y_test
    )

ada_rows = []
for T, res in ada_results.items():
    for split in ["train", "test"]:
        r = res[split]
        ada_rows.append([T, split, r["accuracy"], r["f1"], r["auc"]])

ada_table = pd.DataFrame(ada_rows, columns=["Base learners", "Split", "Accuracy", "F1", "AUC"])
ada_table.round(4)


In [ ]:

dt_model = DecisionTreeClassifier(criterion="entropy", random_state=42)
dt_model.fit(X_train, y_train)

ada100 = AdaBoostClassifier(n_estimators=100, random_state=42)
ada100.fit(X_train, y_train)

plt.figure(figsize=(6, 5))
for label, scores in {
    "Decision Tree": dt_model.predict_proba(X_test)[:, 1],
    "Random Forest (100)": rf100.predict_proba(X_test)[:, 1],
    "AdaBoost (100)": ada100.predict_proba(X_test)[:, 1],
}.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves on SPAMBASE test set")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_curves.png"), dpi=200)
plt.show()


## Problem 4 - Custom Naive Bayes

In [ ]:

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mush_raw, y_mush_raw, test_size=0.25, random_state=42, stratify=y_mush_raw
)

class CategoricalNBManual:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        self.classes_ = sorted(y.unique())
        self.priors_ = {c: (y == c).mean() for c in self.classes_}
        self.feature_values_ = {col: sorted(X[col].unique()) for col in X.columns}
        self.cond_probs_ = {c: {} for c in self.classes_}

        for c in self.classes_:
            Xc = X[y == c]
            for col in X.columns:
                values = self.feature_values_[col]
                counts = Xc[col].value_counts().to_dict()
                denom = len(Xc) + self.alpha * len(values)
                self.cond_probs_[c][col] = {
                    v: (counts.get(v, 0) + self.alpha) / denom for v in values
                }
        return self

    def predict_proba(self, X):
        all_rows = []
        for _, row in X.iterrows():
            log_probs = {}
            for c in self.classes_:
                lp = math.log(self.priors_[c])
                for col, val in row.items():
                    lp += math.log(self.cond_probs_[c][col][val])
                log_probs[c] = lp

            max_lp = max(log_probs.values())
            exp_shifted = {c: math.exp(v - max_lp) for c, v in log_probs.items()}
            total = sum(exp_shifted.values())
            all_rows.append([exp_shifted[c] / total for c in self.classes_])

        return np.array(all_rows)

    def predict(self, X):
        probs = self.predict_proba(X)
        idx = probs.argmax(axis=1)
        return np.array([self.classes_[i] for i in idx])

manual_nb = CategoricalNBManual(alpha=1.0).fit(X_train_m, y_train_m)
manual_pred = manual_nb.predict(X_test_m)

manual_metrics = {
    "accuracy": accuracy_score(y_test_m, manual_pred),
    "precision": precision_score(y_test_m, manual_pred, pos_label="p"),
    "recall": recall_score(y_test_m, manual_pred, pos_label="p"),
    "f1": f1_score(y_test_m, manual_pred, pos_label="p"),
}
print("Class priors:", manual_nb.priors_)
manual_metrics


In [ ]:

# Package comparison using sklearn CategoricalNB
X_train_enc = pd.DataFrame(index=X_train_m.index)
X_test_enc = pd.DataFrame(index=X_test_m.index)

for col in X_train_m.columns:
    le = LabelEncoder()
    le.fit(pd.concat([X_train_m[col], X_test_m[col]], axis=0))
    X_train_enc[col] = le.transform(X_train_m[col])
    X_test_enc[col] = le.transform(X_test_m[col])

y_le = LabelEncoder()
y_train_enc = y_le.fit_transform(y_train_m)
y_test_enc = y_le.transform(y_test_m)

pkg_nb = CategoricalNB(alpha=1.0)
pkg_nb.fit(X_train_enc, y_train_enc)
pkg_pred = y_le.inverse_transform(pkg_nb.predict(X_test_enc))

pkg_metrics = {
    "accuracy": accuracy_score(y_test_m, pkg_pred),
    "precision": precision_score(y_test_m, pkg_pred, pos_label="p"),
    "recall": recall_score(y_test_m, pkg_pred, pos_label="p"),
    "f1": f1_score(y_test_m, pkg_pred, pos_label="p"),
}

comparison = pd.DataFrame([
    ["Custom Naive Bayes", *manual_metrics.values()],
    ["Package CategoricalNB", *pkg_metrics.values()],
], columns=["Model", "Accuracy", "Precision", "Recall", "F1"])

comparison.round(4)
